In [27]:
import pymysql
from typing import Optional
from DATA.stock_invest_function import get_db_host
import pymysql
import pandas as pd
import numpy as np
import re

In [28]:
def fetch_fs_data_by_ticker(db_info: dict,
                            ticker: str,
                            table_name: str = "korea_fs_data_from_DART") -> pd.DataFrame:
    """
    특정 ticker의 재무제표 데이터를 DB에서 조회.
    ticker가 존재하지 않을 경우 메시지 출력 후 빈 DataFrame 반환.
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        # 먼저 ticker 존재 여부 확인
        check_sql = f"SELECT COUNT(*) AS cnt FROM {table_name} WHERE ticker = %s"
        with conn.cursor() as cur:
            cur.execute(check_sql, (ticker,))
            result = cur.fetchone()
            cnt = result[0]

        if cnt == 0:
            print(f"[INFO] ticker '{ticker}' 는(은) 데이터베이스에 존재하지 않습니다.")
            return pd.DataFrame()   # 빈 DF 반환

        # ticker 존재 → 실제 데이터 조회
        query = f"""
            SELECT
                corp_code,
                bsns_year,
                reprt_code,
                quarter,
                account_id,
                sj_div,
                sj_nm,
                account_nm,
                thstrm_nm,
                thstrm_amount,
                report_date,
                ticker
            FROM {table_name}
            WHERE ticker = %s
            ORDER BY
                bsns_year,
                reprt_code,
                sj_div,
                account_nm
        """

        df = pd.read_sql(query, conn, params=[ticker])
        return df

    finally:
        conn.close()

def adjust_quarterly_from_index(df: pd.DataFrame, value_cols: list) -> pd.DataFrame:
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    original_index_name = df.index.name

    df["year"] = df.index.year
    df["quarter"] = df.index.month.map({3: "Q1", 6: "Q2", 9: "Q3", 12: "Q4"})

    adjusted_chunks = []

    for year, grp in df.groupby("year"):
        grp = grp.sort_index()

        q1 = grp[grp["quarter"] == "Q1"]
        q2 = grp[grp["quarter"] == "Q2"]
        q3 = grp[grp["quarter"] == "Q3"]
        q4 = grp[grp["quarter"] == "Q4"]

        if len(q4) > 0:
            q4 = q4.copy()

            for col in value_cols:
                if col not in grp.columns:
                    continue

                fy = q4[col].iloc[0]  # 12월 값(FY라고 가정)
                if pd.isna(fy):
                    continue

                prev_sum = (
                    q1[col].fillna(0).sum()
                    + q2[col].fillna(0).sum()
                    + q3[col].fillna(0).sum()
                )

                q4[col] = fy - prev_sum   # ★ 여기서 딱 Q4만 수정

            adjusted_chunks.extend([q1, q2, q3, q4])
        else:
            # 4Q(12월)가 없으면 그 연도는 그대로
            adjusted_chunks.append(grp)

    result = pd.concat(adjusted_chunks).sort_index()
    result = result.drop(columns=["year", "quarter"])
    result.index.name = original_index_name
    return result


from typing import Optional, List

def cumulative_to_quarterly(df: pd.DataFrame,
                            value_cols: List[str],
                            exclude_date: Optional[str] = "2025-12-31") -> pd.DataFrame:
    """
    연도별 누적값(1Q,2Q,3Q,4Q)을 순수 분기값으로 변환.
    df: index가 날짜(분기말)인 DataFrame
    value_cols: 변환할 숫자 컬럼 리스트
    exclude_date: 제외할 날짜 (예: '2025-12-31')
    """
    out = df.copy()
    out.index = pd.to_datetime(out.index)
    out = out.sort_index()

    # 특정 날짜 제거
    if exclude_date is not None:
        out = out.loc[out.index != pd.to_datetime(exclude_date)].copy()

    years = out.index.year

    for col in value_cols:
        def _to_quarterly(s: pd.Series) -> pd.Series:
            s = s.sort_index()
            q = s.diff()
            if len(s) > 0:
                q.iloc[0] = s.iloc[0]
            return q

        out[col] = (
            out[col]
            .groupby(years)
            .apply(_to_quarterly)
            .reset_index(level=0, drop=True)
        )

    return out


# 0으로 나누는 경우 inf가 생기지 않도록 float 변환 + 나누기 후 정리
def safe_divide(num, den):
    result = num / den
    # 0으로 나눠서 생긴 inf/-inf 를 NaN으로 처리
    result = result.replace([np.inf, -np.inf], np.nan)
    return result



account_groups = {

    # -------------------------------------------------
    # 1) 매출액 (Revenue)
    # -------------------------------------------------
    "revenue": [
        "ifrs_Revenue",
        "ifrs-full_Revenue",
    ],

    # -------------------------------------------------
    # 2) 매출총이익 (Gross Profit)
    # -------------------------------------------------
    "gross_profit": [
        "ifrs_GrossProfit",
        "ifrs-full_GrossProfit",
    ],

    # -------------------------------------------------
    # 3) 영업이익 (Operating Income)
    # -------------------------------------------------
    "operating_income": [
        "dart_OperatingIncomeLoss",
    ],

    # -------------------------------------------------
    # 4) 당기순이익 (Net Income) - 총당기순이익
    #     손익계산서 기준 전체 당기순이익 + CF용 당기순이익 포함
    # -------------------------------------------------
    "net_income_total": [
        "ifrs_ProfitLoss",
        "ifrs-full_ProfitLoss",
        "dart_ProfitLossForStatementOfCashFlows",
    ],

    # -------------------------------------------------
    # 4-1) 계속사업 관련 손익 (계속영업이익, 계속사업 법인세 등)
    # -------------------------------------------------
    "continuing_operations": [
        "ifrs_ProfitLossBeforeTax",
        "ifrs-full_ProfitLossBeforeTax",
    ],

    # -------------------------------------------------
    # 4-2) 계속사업 법인세 등
    # -------------------------------------------------
    "income_tax": [
        "ifrs_IncomeTaxExpenseContinuingOperations",             # 계속사업 법인세비용
        "ifrs-full_IncomeTaxExpenseContinuingOperations",        # (full IFRS) 계속사업 법인세비용
    ],


    # -------------------------------------------------
    # 5) 당기순이익 - 지배기업 소유주 귀속
    # -------------------------------------------------
    "net_income_parent": [
        "ifrs_ProfitLossAttributableToOwnersOfParent",
        "ifrs-full_ProfitLossAttributableToOwnersOfParent",
    ],

    # -------------------------------------------------
    # 6) 당기순이익 - 비지배지분 귀속
    # -------------------------------------------------
    "net_income_nci": [
        "ifrs_ProfitLossAttributableToNoncontrollingInterests",
        "ifrs-full_ProfitLossAttributableToNoncontrollingInterests",
    ],

    "discontinued_pl_accounts" : ["ifrs_ProfitLossFromDiscontinuedOperations"],
}

account_groups_bs = {

    # -------------------------------------------------
    # 1) 자산총계 (Total Assets)
    # -------------------------------------------------
    "assets_total": [
        "ifrs_Assets",              # IFRS 전체 자산
        "ifrs-full_Assets",         # IFRS full
    ],

    "cash": [
        "ifrs_CashAndCashEquivalents",
        "ifrs-full_CashAndCashEquivalents",              # IFRS 현금
    ],

    # -------------------------------------------------
    # 2) 유동자산 (Current Assets)
    # -------------------------------------------------
    "current_assets": [
        "ifrs_CurrentAssets",
        "ifrs-full_CurrentAssets",
    ],

    # -------------------------------------------------
    # 3) 재고자산 (Inventories)
    # -------------------------------------------------
    "inventories": [
        "ifrs_Inventories",
        "ifrs-full_Inventories",
    ],

    # -------------------------------------------------
    # 4) 유형자산 (Property, Plant and Equipment)
    # -------------------------------------------------
    "ppe": [
        "ifrs_PropertyPlantAndEquipment",
        "ifrs-full_PropertyPlantAndEquipment",
    ],

    # -------------------------------------------------
    # 5) 부채총계 (Total Liabilities)
    # -------------------------------------------------
    "liabilities_total": [
        "ifrs_Liabilities",
        "ifrs-full_Liabilities",
    ],

    # -------------------------------------------------
    # 6) 유동부채 (Current Liabilities)
    # -------------------------------------------------
    "current_liabilities": [
        "ifrs_CurrentLiabilities",
        "ifrs-full_CurrentLiabilities",
    ],

    # -------------------------------------------------
    # 7) 비유동부채 (Noncurrent Liabilities)
    # -------------------------------------------------
    "noncurrent_liabilities": [
        "ifrs_NoncurrentLiabilities",
        "ifrs-full_NoncurrentLiabilities",
    ],
}

account_groups_cf = {

    # -------------------------------------------------
    # 1) 영업활동 현금흐름
    # -------------------------------------------------
    "cf_operating": [
        "ifrs_CashFlowsFromUsedInOperatingActivities",
        "ifrs-full_CashFlowsFromUsedInOperatingActivities",
    ],

    # 영업활동 조정항목 (당기순이익 → 영업CF로 reconcile)
    "cf_adjustments": [
        "ifrs_AdjustmentsForReconcileProfitLoss",
        "ifrs-full_CashFlowsFromUsedInOperations",  # Operations 기준도 함께
    ],

    # -------------------------------------------------
    # 2) 투자활동 현금흐름
    # -------------------------------------------------
    "cf_investing": [
        "ifrs_CashFlowsFromUsedInInvestingActivities",
        "ifrs-full_CashFlowsFromUsedInInvestingActivities",
    ],

    # -------------------------------------------------
    # 3) 재무활동 현금흐름
    # -------------------------------------------------
    "cf_financing": [
        "ifrs_CashFlowsFromUsedInFinancingActivities",
        "ifrs-full_CashFlowsFromUsedInFinancingActivities",
    ],

    # -------------------------------------------------
    # 4) 법인세 납부(환급) - 영업활동 분류
    # -------------------------------------------------
    "cf_tax_operating": [
        "ifrs_IncomeTaxesPaidRefundClassifiedAsOperatingActivities",
        "ifrs-full_IncomeTaxesPaidRefundClassifiedAsOperatingActivities",
    ],

    # -------------------------------------------------
    # 5) 이자수익/이자비용 - 영업활동 분류
    # -------------------------------------------------
    "cf_interest_received": [
        "ifrs_InterestReceivedClassifiedAsOperatingActivities",
        "ifrs-full_InterestReceivedClassifiedAsOperatingActivities",
    ],
    "cf_interest_paid": [
        "ifrs_InterestPaidClassifiedAsOperatingActivities",
        "ifrs-full_InterestPaidClassifiedAsOperatingActivities",
    ],

    # -------------------------------------------------
    # 6) 배당수익/배당금 지급
    # -------------------------------------------------
    "cf_dividends_received": [
        "ifrs_DividendsReceivedClassifiedAsOperatingActivities",
        "ifrs-full_DividendsReceivedClassifiedAsOperatingActivities",
    ],
    "cf_dividends_paid": [
        "ifrs_DividendsPaidClassifiedAsFinancingActivities",
        "ifrs-full_DividendsPaidClassifiedAsFinancingActivities",
        "ifrs_DividendsPaid",
        "ifrs-full_DividendsPaid",
    ],

    # -------------------------------------------------
    # 7) 차입/상환 (재무활동 디테일)
    # -------------------------------------------------
    "cf_borrowings": [
        "ifrs_ProceedsFromBorrowingsClassifiedAsFinancingActivities",
        "ifrs-full_ProceedsFromBorrowingsClassifiedAsFinancingActivities",
    ],
    "cf_repayments": [
        "ifrs_RepaymentsOfBorrowingsClassifiedAsFinancingActivities",
        "ifrs-full_RepaymentsOfBorrowingsClassifiedAsFinancingActivities",
    ],

    # -------------------------------------------------
    # 8) 기초/기말 현금 및 현금성자산, 증감, 환율효과
    # -------------------------------------------------
    "cf_beginning_cash": [
        "dart_CashAndCashEquivalentsAtBeginningOfPeriodCf",
    ],
    "cf_ending_cash": [
        "dart_CashAndCashEquivalentsAtEndOfPeriodCf",
    ],
    "cf_increase_decrease_cash": [
        "ifrs_IncreaseDecreaseInCashAndCashEquivalents",
        "ifrs-full_IncreaseDecreaseInCashAndCashEquivalents",
    ],
    "cf_fx_effect": [
        "ifrs_EffectOfExchangeRateChangesOnCashAndCashEquivalents",
        "ifrs-full_EffectOfExchangeRateChangesOnCashAndCashEquivalents",
    ],

    # -------------------------------------------------
    # 9) 기타: 단기예금/투자, 리스상환, 정부보조금 등
    #    (필요시 나중에 세분화해서 쓰실 수 있게 모아둠)
    # -------------------------------------------------
    "cf_other_investing": [
        "ifrs-full_ProceedsFromSalesOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities",
        "ifrs-full_PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities",
        "ifrs-full_ProceedsFromSalesOfIntangibleAssetsClassifiedAsInvestingActivities",
        "ifrs-full_PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities",
        "ifrs-full_ProceedsFromGovernmentGrantsClassifiedAsInvestingActivities",
        "ifrs-full_OtherInflowsOutflowsOfCashClassifiedAsInvestingActivities",
    ],
    "cf_other_financing": [
        "ifrs-full_PaymentsOfFinanceLeaseLiabilitiesClassifiedAsFinancingActivities",
        "ifrs-full_PaymentsOfLeaseLiabilitiesClassifiedAsFinancingActivities",
        "ifrs-full_OtherInflowsOutflowsOfCashClassifiedAsFinancingActivities",
        "ifrs-full_SaleOrIssueOfTreasuryShares",
        "ifrs-full_IncreaseDecreaseThroughSharebasedPaymentTransactions",
    ],
}

def extract_is_group(df: pd.DataFrame,
                     group_name: str,
                     account_groups: dict) -> pd.DataFrame:
    """
    손익계산서(IS)에서 account_groups[group_name] 에 속하는
    account_id 행만 추출해서 반환.
    """
    ids = account_groups.get(group_name, [])
    if not ids:
        raise ValueError(f"{group_name} 에 대한 account_id 리스트가 비어 있습니다.")

    mask = ((df["sj_div"] == "IS") | (df["sj_div"] == "CIS")) & df["account_id"].isin(ids)
    out = df.loc[mask].copy()
    return out


def extract_bs_group(df: pd.DataFrame,
                     group_name: str,
                     account_groups_bs: dict) -> pd.DataFrame:

    ids = account_groups_bs.get(group_name, [])
    if not ids:
        raise ValueError(f"{group_name} 에 대한 account_id 리스트가 비어 있습니다.")

    mask = (df["sj_div"] == "BS") & df["account_id"].isin(ids)
    return df.loc[mask].copy()

def extract_cf_group(df: pd.DataFrame,
                     group_name: str,
                     account_groups_cf: dict) -> pd.DataFrame:
    """
    현금흐름표(CF)에서 account_groups_cf[group_name] 에 속하는
    account_id 행만 추출해서 반환.
    """
    ids = account_groups_cf.get(group_name, [])
    if not ids:
        raise ValueError(f"{group_name} 에 대한 account_id 리스트가 비어 있습니다.")

    mask = (df["sj_div"] == "CF") & df["account_id"].isin(ids)
    out = df.loc[mask].copy()
    return out

def normalize_name(name: str) -> str:
    """
    계정명을 정규화하여 유사 항목을 하나의 대표 이름으로 통합하는 함수.
    - 공백 제거
    - 특수문자 제거
    - 숫자 제거
    - 동의어/변형어 통합
    """
    if pd.isna(name):
        return "unknown"

    n = str(name).strip()

    # 1) 특수문자·공백 제거
    n = re.sub(r"[\s\(\)\[\]\/]", "", n)

    # 2) 숫자 제거
    n = re.sub(r"[0-9]+", "", n)

    # 3) 주요 패턴 통합
    replacements = {
        "배당금지급": "배당금지급",
        "배당금의지급": "배당금지급",
        "배당의지급": "배당금지급",
        "배당금": "배당금지급",

        "영업이익손실": "영업이익",
        "영업이익": "영업이익",

        "법인세비용차감전순이익손실": "법인세차감전순이익",
        "법인세비용차감전순이익": "법인세차감전순이익",
        "법인세차감전순이익": "법인세차감전순이익",
    }

    for key, val in replacements.items():
        if key in n:
            return val

    return n  # 기본값


def make_pivot(df: pd.DataFrame) -> pd.DataFrame:
    """
    - account_nm을 정규화하여, 유사 계정들이 하나의 컬럼으로 통합되도록 개선한 pivot 함수
    - report_date → index
    - thstrm_amount → values
    """

    if df.empty:
        raise ValueError("입력된 DataFrame이 비어 있습니다.")

    df = df.copy()
    df["report_date"] = pd.to_datetime(df["report_date"])

    # 1) account_nm 정규화 (유사 계정명 자동 통합)
    df["account_nm_rep"] = df["account_nm"].apply(normalize_name)

    # 2) pivot 생성
    pivot_df = df.pivot_table(
        index="report_date",
        columns="account_nm_rep",
        values="thstrm_amount",
        aggfunc="sum"     # 동일 그룹은 합쳐서 하나의 컬럼으로 통합
    )

    # 3) 컬럼 정렬
    pivot_df = pivot_df.reindex(sorted(pivot_df.columns), axis=1)

    return pivot_df


def adjust_quarterly_q4_only(df: pd.DataFrame, value_cols: list) -> pd.DataFrame:
    """
    1Q, 2Q, 3Q는 원래 값 그대로 두고,
    4Q(12월)만 `FY - (Q1+Q2+Q3)`로 조정하는 함수.

    전제:
      - df.index: DatetimeIndex (분기말 날짜)
      - value_cols: 조정할 수치 칼럼 리스트
    """
    out = df.copy()
    out.index = pd.to_datetime(out.index)

    # 연도별로 처리
    for year in sorted(out.index.year.unique()):
        mask_year = out.index.year == year
        sub = out.loc[mask_year].sort_index()

        q1_idx = sub.index[sub.index.month == 3]
        q2_idx = sub.index[sub.index.month == 6]
        q3_idx = sub.index[sub.index.month == 9]
        q4_idx = sub.index[sub.index.month == 12]

        # 4분기가 없으면 그 해는 스킵
        if len(q4_idx) == 0:
            continue

        # 4분기 행(여러 개라면 첫 번째만 사용한다고 가정)
        q4_i = q4_idx[0]

        for col in value_cols:
            if col not in out.columns:
                continue

            fy = out.loc[q4_i, col]
            if pd.isna(fy):
                continue

            prev_sum = (
                out.loc[q1_idx, col].fillna(0).sum()
                + out.loc[q2_idx, col].fillna(0).sum()
                + out.loc[q3_idx, col].fillna(0).sum()
            )

            # ✅ 오직 4분기 값만 조정
            out.loc[q4_i, col] = fy - prev_sum

    return out

from difflib import SequenceMatcher

def merge_similar_columns_smart(df: pd.DataFrame, threshold: float = 0.75) -> pd.DataFrame:
    """
    pivot table에서 비슷한 계정명(예: 'A', 'A(손실)') 또는
    한국어 계정명이 유사한 경우 자동 병합하여 하나의 컬럼으로 통일.

    threshold : 0~1, 문자열 유사도 임계값
    """
    out = df.copy()

    # 1) 전처리 후 key 생성
    def normalize(col):
        col = str(col)
        col = re.sub(r"\(.*?\)", "", col)   # 괄호 제거
        col = col.replace(" ", "")          # 공백 제거
        col = re.sub(r"[^가-힣A-Za-z]", "", col)   # 특수문자 제거
        return col

    columns = list(out.columns)
    norm_cols = [normalize(c) for c in columns]

    # 2) 그룹핑 수행
    groups = {}
    used = set()

    for i, base in enumerate(norm_cols):
        if i in used:
            continue

        groups[columns[i]] = [columns[i]]
        used.add(i)

        # 다른 컬럼들과 유사도 검사
        for j in range(i + 1, len(columns)):
            if j in used:
                continue

            ratio = SequenceMatcher(None, base, norm_cols[j]).ratio()
            if ratio >= threshold:
                groups[columns[i]].append(columns[j])
                used.add(j)

    # 3) 그룹 내부 병합
    for base_col, cols in groups.items():
        for col in cols:
            if col == base_col:
                continue
            out[base_col] = out[base_col].combine_first(out[col])
            out = out.drop(columns=[col])

    return out


def fetch_fs_from_db(db_info: dict,
                     symbol: str,
                     table_name: str = "korea_fs_data") -> pd.DataFrame:
    """
    MariaDB의 korea_fs_data 테이블에서
    특정 종목(symbol)의 재무 데이터를 읽어오는 함수.

    Parameters
    ----------
    db_info : dict
        {
            "host": ...,
            "port": ...,
            "user": ...,
            "password": ...,
            "database": ...
        }
    symbol : str
        예) 'A000030'
    table_name : str
        기본값 'korea_fs_data'

    Returns
    -------
    pd.DataFrame
        symbol, company_name, date, indicator, value 컬럼 포함.
        데이터가 없으면 빈 DataFrame 반환.
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        autocommit=False,
    )

    try:
        sql = f"""
            SELECT
                symbol,
                company_name,
                date,
                indicator,
                value
            FROM {table_name}
            WHERE symbol = %s
            ORDER BY date, indicator
        """
        df = pd.read_sql(sql, conn, params=(symbol,))
        return df

    finally:
        conn.close()

def df_to_long_format(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """
    wide-format 재무 데이터를 long-format(date, ticker, indicator, value) 형태로 변환
    """
    if df.empty:
        raise ValueError("입력된 DataFrame(df)이 비어 있습니다.")

    # 1) index를 date로 사용하기 위해 reset_index
    df2 = df.copy().reset_index()

    # 2) long-format 변환 (melt)
    long_df = pd.melt(
        df2,
        id_vars=["date"],                    # 날짜는 고정
        var_name="indicator",                # 기존 컬럼명이 indicator
        value_name="value"                   # 값은 value
    )

    # 3) ticker 입력값 추가
    long_df["ticker"] = ticker

    # 4) 컬럼 정렬
    long_df = long_df[["date", "ticker", "indicator", "value"]]

    return long_df


In [59]:
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user' : 'stox7412',
    'password' : 'Apt106503!~',
    'database': 'investar'
}

In [96]:
ticker = "006400"
symbol = 'A' + ticker

df = fetch_fs_data_by_ticker(db_info, ticker)

if df.empty:
    print("데이터 없음")
else:
    display(df.head())

# 1) report_date 를 datetime 으로 변환 (안 되어 있으면)
df["report_date"] = pd.to_datetime(df["report_date"])

# 1) 매출액 관련 계정만 필터링

#
ni_total_df = extract_is_group(df, "net_income_total", account_groups)
# 2) 계속사업 관련 손익 추출
cont_ops_df = extract_is_group(df, "continuing_operations", account_groups)
# 3) 지배주주 귀속 순이익
ni_parent_df = extract_is_group(df, "net_income_parent", account_groups)
# 4) 비지배지분 귀속 순이익
ni_nci_df = extract_is_group(df, "net_income_nci", account_groups)
# 5) 법인세 비용
tax_df = extract_is_group(df, "income_tax", account_groups)
disc_df = extract_is_group(df, "discontinued_pl_accounts", account_groups)
rev_df = df[df["account_id"].isin(account_groups["revenue"])]
gp_df = df[df["account_id"].isin(account_groups["gross_profit"])]
op_df = df[df["account_id"].isin(account_groups["operating_income"])]
# tax_df = df[df["account_id"].isin(account_groups["net_income_total"])]
# ni_parent = df[df["account_id"].isin(account_groups["net_income_parent"])]
# ni_nci = df[df["account_id"].isin(account_groups["net_income_nci"])]


# 2) 대차대조표 계정만 필터링
assets_df = extract_bs_group(df, "assets_total", account_groups_bs)
cash_df = extract_bs_group(df, "cash", account_groups_bs)
current_assets_df = extract_bs_group(df, "current_assets", account_groups_bs)
# inventories_df = extract_bs_group(df, "inventories", account_groups_bs)
ppe_df = extract_bs_group(df, "ppe", account_groups_bs)
liabilities_df = extract_bs_group(df, "liabilities_total", account_groups_bs)
current_liab_df = extract_bs_group(df, "current_liabilities", account_groups_bs)
noncurrent_liab_df = extract_bs_group(df, "noncurrent_liabilities", account_groups_bs)

# 3) 현금흐름표 계정만 필터링

cf_op_df   = extract_cf_group(df, "cf_operating", account_groups_cf)
cf_inv_df  = extract_cf_group(df, "cf_investing", account_groups_cf)
cf_fin_df  = extract_cf_group(df, "cf_financing", account_groups_cf)
cf_tax_df  = extract_cf_group(df, "cf_tax_operating", account_groups_cf)
cf_div_df  = extract_cf_group(df, "cf_dividends_paid", account_groups_cf)

rev_pivot = make_pivot(rev_df)
gp_pivot = make_pivot(gp_df)
op_pivot = make_pivot(op_df)
cont_ops_pivot = make_pivot(cont_ops_df)
tax_pivot = make_pivot(tax_df)

# pivot 합친 뒤
is_table = pd.concat([rev_pivot, gp_pivot, op_pivot, cont_ops_pivot, tax_pivot], axis=1)
is_table.index.name = 'report_date'

# 🔹 비슷한 이름의 컬럼(예: …(손실))을 한 컬럼으로 병합
is_table = merge_similar_columns_smart(is_table)

# Q4 조정
value_cols = is_table.columns.tolist()
is_table_adj = adjust_quarterly_q4_only(is_table, value_cols)

print(is_table_adj.columns.tolist())  # 확인용

# 이제 컬럼이 5개가 되었을 것
is_table_adj.columns = ['매출액', '매출총이익', '영업이익', '법인세비용차감전순이익', '법인세비용']
is_table_adj['당기순이익'] = (
    is_table_adj['법인세비용차감전순이익'] - is_table_adj['법인세비용']
)

assets_pivot = make_pivot(assets_df)
cash_pivot = make_pivot(cash_df)
current_assets_pivot  = make_pivot(current_assets_df)
inventories_pivot = make_pivot(inventories_df)
ppe_pivot  = make_pivot(ppe_df)
liabilities_pivot  = make_pivot(liabilities_df)
current_liab_pivot  = make_pivot(current_liab_df)
noncurrent_liab_pivot  = make_pivot(noncurrent_liab_df)

bs_table = pd.concat([assets_pivot, cash_pivot, current_assets_pivot,
                   liabilities_pivot, current_liab_pivot, noncurrent_liab_pivot], axis=1)

bs_table.columns= ['자산총계', '현금및현금성자산', '유동자산', '부채총계', '유동부채', '비유동부채' ]
bs_table['자본총계'] = bs_table['자산총계'] - bs_table['부채총계']

cf_op_pivot = make_pivot(cf_op_df)
cf_inv_pivot = make_pivot(cf_inv_df)
cf_fin_pivot = make_pivot(cf_fin_df)
cf_tax_pivot = make_pivot(cf_tax_df)
cf_div_pivot = make_pivot(cf_div_df)

cf_table = pd.concat([cf_op_pivot, cf_inv_pivot, cf_fin_pivot, cf_tax_pivot, cf_div_pivot], axis=1)
value_cols = cf_table.columns.tolist()
cf_table = cumulative_to_quarterly(cf_table, value_cols)

fs_df = pd.concat([is_table_adj, bs_table, cf_table], axis=1)

fs_df.index.name = 'date'
#
# 2) 인덱스가 datetime 이 아니면 변환
fs_df.index = pd.to_datetime(fs_df.index)
#
# 3) 날짜 기준 정렬
fs_df = fs_df.sort_index()

# 모든 수치 컬럼을 1/1000로 (천원 단위 맞추기)
fs_df_scaled = fs_df / 1000.0

,corp_code,bsns_year,reprt_code,quarter,account_id,sj_div,sj_nm,account_nm,thstrm_nm,thstrm_amount,report_date,ticker
0,00126362,2015,11011,FY,ifrs_OtherNoncurrentNonfinancialAssets,BS,재무상태표,기타비유동자산,제 46 기,1.384722e+11,2015-12-31,006400
1,00126362,2015,11011,FY,ifrs_OtherCurrentNonfinancialAssets,BS,재무상태표,기타유동자산,제 46 기,7.871019e+10,2015-12-31,006400
2,00126362,2015,11011,FY,ifrs_OtherEquityInterest,BS,재무상태표,기타자본,제 46 기,-1.084867e+10,2015-12-31,006400
3,00126362,2015,11011,FY,dart_ElementsOfOtherStockholdersEquity,BS,재무상태표,기타자본구성요소,제 46 기,5.802145e+12,2015-12-31,006400
4,00126362,2015,11011,FY,dart_OtherCapitalSurplus,BS,재무상태표,기타자본잉여금,제 46 기,5.031244e+12,2015-12-31,006400


['매출', '매출총이익', '영업손익', '법인세비용차감전순손익', '법인세비용']


In [97]:
def make_fs_pivot(df: pd.DataFrame, item_list: list) -> pd.DataFrame:
    """
    korea_fs_data 형태(df)에서 item_list에 포함된 indicator만 추출하여
    date(index) × indicator(columns) pivot_table 생성.

    Parameters
    ----------
    df : DataFrame (columns: symbol, company_name, date, indicator, value)
    item_list : list of indicators to include

    Returns
    -------
    pivot_df : DataFrame (index=date, columns=item_list)
    """

    if df.empty:
        raise ValueError("입력 df가 비어 있습니다.")

    # 날짜 변환
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    # item_list 항목만 필터링
    df_filtered = df[df["indicator"].isin(item_list)].copy()

    if df_filtered.empty:
        print("⚠️ item_list에 해당하는 데이터가 없습니다.")
        return pd.DataFrame()

    # pivot 생성
    pivot_df = df_filtered.pivot_table(
        index="date",
        columns="indicator",
        values="value",
        aggfunc="first"
    ).sort_index()

    # item_list 순서대로 정렬
    pivot_df = pivot_df.reindex(columns=item_list)

    return pivot_df

rename_fn = {
    '매출액(천원)': '매출액',
    '매출총이익(천원)': '매출총이익',
    '영업이익(천원)': '영업이익',
    '계속사업이익(천원)': '계속사업이익',
    '당기순이익(천원)': '당기순이익',
    '총자산(천원)': '자산총계',
    '유동자산(천원)': '유동자산',
    '재고자산(천원)': '재고자산',
    '총자본(천원)': '자본총계',
    '배당금지급(영업,투자,재무)(천원)': '배당금',
}

rename_fs = {
    '매출액': '매출액',
    '매출총이익': '매출총이익',
    '영업이익': '영업이익',
    '계속사업이익': '계속사업이익',
    '당기순이익': '당기순이익',
    '자산총계': '자산총계',
    '유동자산': '유동자산',
    '재고자산': '재고자산',
    '자본총계': '자본총계',
    '배당금지급': '배당금',    # ← 여기서 배당금지급을 배당금으로 통일
}

In [98]:
item_list = [
    '매출액(천원)', '매출총이익(천원)', '영업이익(천원)',
    '계속사업이익(천원)', '당기순이익(천원)',
    '총자산(천원)', '유동자산(천원)', '재고자산(천원)',
    '총자본(천원)', '배당금지급(영업,투자,재무)(천원)'
]

df = fetch_fs_from_db(db_info, symbol)

fn_df = make_fs_pivot(df, item_list)

fn_df_u = fn_df.rename(columns=rename_fn)
fs_df_u = fs_df_scaled.rename(columns=rename_fs)


# 최종적으로 쓰고 싶은 공통 컬럼 목록
common_cols = [
    '매출액',
    '매출총이익',
    '영업이익',
    '계속사업이익',
    '당기순이익',
    '자산총계',
    '유동자산',
    '재고자산',
    '자본총계',
    '배당금',
]

# 인덱스가 date(datetime)라고 가정
fn_df_u = fn_df_u.sort_index()
fs_df_u = fs_df_u.sort_index()

# 공통 컬럼만 남기기 (없는 컬럼은 자동으로 스킵)
fn_df_u = fn_df_u[[c for c in common_cols if c in fn_df_u.columns]]
fs_df_u = fs_df_u[[c for c in common_cols if c in fs_df_u.columns]]

# 날짜 전체 범위(합집합)
all_dates = fn_df_u.index.union(fs_df_u.index)

fn_all = fn_df_u.reindex(all_dates)
fs_all = fs_df_u.reindex(all_dates)

# ✅ fn_df 값이 우선, NaN인 부분만 fs_df 값으로 채우기
merged_df = fn_all.combine_first(fs_all)

# 보기 좋게 정렬
merged_df = merged_df[ [c for c in common_cols if c in merged_df.columns] ]
merged_df_resize =  merged_df[['매출액', '매출총이익', '영업이익', '당기순이익', '자산총계', '자본총계', '배당금']]

merged_df_resize['매출총이익_ttm'] = merged_df_resize['매출총이익'].rolling(4).sum()
merged_df_resize['당기순이익_ttm'] = merged_df_resize['당기순이익'].rolling(4).sum()
merged_df_resize['영업이익_ttm']  = merged_df_resize['영업이익'].rolling(4).sum()
merged_df_resize['매출액_ttm']    = merged_df_resize['매출액'].rolling(4).sum()
#
# 1) 전년 동기대비(4분기 전) 자본총계
merged_df_resize['자본총계_lag4'] = merged_df_resize['자본총계'].shift(4)
merged_df_resize['자산총계_lag4'] = merged_df_resize['자산총계'].shift(4)
#
# 2) 자본총계 평균 (기초+기말)/2 개념
merged_df_resize['자본총계_평균'] = (merged_df_resize['자본총계'] + merged_df_resize['자본총계_lag4']) / 2
merged_df_resize['자산총계_평균'] = (merged_df_resize['자산총계'] + merged_df_resize['자산총계_lag4']) / 2
#
# 1) GPM: Gross Profit Margin
merged_df_resize['GP_ttm'] = safe_divide(merged_df_resize['매출총이익_ttm'], merged_df_resize['매출액_ttm'])
#
# 2) OPM: Operating Profit Margin
merged_df_resize['OPM_ttm'] = safe_divide(merged_df_resize['영업이익_ttm'], merged_df_resize['매출액_ttm'])
#
# 3) NIM: Net Income Margin
merged_df_resize['NIM_ttm'] = safe_divide(merged_df_resize['당기순이익_ttm'], merged_df_resize['매출액_ttm'])
#
# 1) GPM: Gross Profit Margin
merged_df_resize['GPM_ttm'] = safe_divide(merged_df_resize['매출총이익_ttm'], merged_df_resize['매출액_ttm'])
#
# 2) OPM: Operating Profit Margin
merged_df_resize['OPM_ttm'] = safe_divide(merged_df_resize['영업이익_ttm'], merged_df_resize['매출액_ttm'])
#
# 3) NIM: Net Income Margin
merged_df_resize['NIM_ttm'] = safe_divide(merged_df_resize['당기순이익_ttm'], merged_df_resize['매출액_ttm'])
#
# 4) ROA: Return on Assets
merged_df_resize['ROA'] = safe_divide(merged_df_resize['당기순이익_ttm'], merged_df_resize['자산총계_평균'])
#
# 5) ROE: Return on Equity
merged_df_resize['ROE'] = safe_divide(merged_df_resize['당기순이익_ttm'], merged_df_resize['자본총계_평균'])
#
# 6) 배당성향: payout_ratio
merged_df_resize['payout_ratio'] = safe_divide(merged_df_resize['배당금'], merged_df_resize['당기순이익_ttm'])


In [99]:
merged_df_resize

,매출액,매출총이익,영업이익,당기순이익,자산총계,자본총계,배당금,매출총이익_ttm,당기순이익_ttm,영업이익_ttm,...,자본총계_lag4,자산총계_lag4,자본총계_평균,자산총계_평균,GPM_ttm,OPM_ttm,NIM_ttm,ROA,ROE,payout_ratio
date,,,,,,,,,,,,,,,,,,,,,
2004-03-31,1.444871e+09,3.342299e+08,1.702532e+08,2.308024e+08,5.498622e+09,3.971802e+09,-1.114464e+08,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-06-30,1.644207e+09,3.547248e+08,1.813501e+08,2.434546e+08,5.840273e+09,4.202404e+09,9.892000e+03,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-09-30,1.535285e+09,3.095190e+08,1.339164e+08,2.333047e+08,6.047759e+09,4.470207e+09,-2.247104e+07,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2004-12-31,1.497414e+09,1.683630e+08,-4.215502e+07,3.418712e+07,5.664486e+09,4.310042e+09,2.655380e+05,1.166837e+09,7.417489e+08,4.433647e+08,...,NaN,NaN,NaN,NaN,0.190604,0.072424,0.121166,NaN,NaN,0.000358
2005-03-31,1.309133e+09,1.183059e+08,-4.348593e+07,5.411874e+07,5.517036e+09,4.277950e+09,-1.092105e+08,9.509126e+08,5.650652e+08,2.296255e+08,...,3.971802e+09,5.498622e+09,4.124876e+09,5.507829e+09,0.158855,0.038360,0.094397,0.102593,0.136990,-0.193271
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-09-30,3.935667e+09,7.578010e+08,1.195666e+08,2.304005e+08,3.814488e+10,2.113521e+10,0.000000e+00,3.360207e+09,1.311591e+09,8.097161e+08,...,1.922402e+10,3.354676e+10,2.017961e+10,3.584582e+10,0.192274,0.046333,0.075050,0.036590,0.064996,0.000000
2024-12-31,3.754458e+09,5.711657e+08,-2.815832e+08,-2.427336e+08,4.059734e+10,2.156724e+10,-2.742162e+06,3.093564e+09,5.755124e+08,2.734476e+08,...,1.990725e+10,3.403886e+10,2.073724e+10,3.731810e+10,0.186446,0.016480,0.034686,0.015422,0.027753,-0.004765
2025-03-31,3.176818e+09,2.023306e+08,-5.434537e+08,-2.159552e+08,4.071007e+10,2.153695e+10,0.000000e+00,2.452897e+09,7.286856e+07,-4.724269e+08,...,2.060509e+10,3.545063e+10,2.107102e+10,3.808035e+10,0.164042,-0.031594,0.004873,0.001914,0.003458,0.000000


In [100]:
fs_df.columns.tolist()

['매출액',
 '매출총이익',
 '영업이익',
 '법인세비용차감전순이익',
 '법인세비용',
 '당기순이익',
 '자산총계',
 '현금및현금성자산',
 '유동자산',
 '부채총계',
 '유동부채',
 '비유동부채',
 '자본총계',
 '영업활동현금흐름',
 '투자활동현금흐름',
 '재무활동현금흐름',
 '법인세납부환급',
 '법인세의납부',
 '법인세의지급',
 '법인세의환급지급',
 '법인세환급납부',
 '배당금지급']

In [46]:
fs_df.head(5)

,매출액,매출총이익,영업이익,법인세비용차감전순이익,법인세비용,당기순이익,자산총계,현금및현금성자산,유동자산,재고자산,...,영업활동으로인한현금흐름,영업활동현금흐름,투자활동으로인한현금흐름,투자활동현금흐름,재무활동으로인한현금흐름,재무활동현금흐름,법인세납부액,법인세납부환급,법인세의납부,배당금지급
date,,,,,,,,,,,,,,,,,,,,,
2015-12-31,9.947535e+10,4.558499e+10,3.600993e+10,4.065670e+10,8.024249e+09,3.263245e+10,1.883407e+11,2.188103e+10,1.051381e+11,4.600574e+09,...,NaN,2.928935e+10,NaN,-1.907362e+10,NaN,-1.076013e+10,NaN,7.952909e+09,NaN,1.057866e+10
2016-03-31,2.657482e+10,1.213424e+10,9.935632e+09,1.020923e+10,2.006822e+09,8.202409e+09,1.992433e+11,2.658889e+10,1.138723e+11,4.528640e+09,...,NaN,4.389265e+09,NaN,4.266911e+08,NaN,0.000000e+00,NaN,1.927009e+09,NaN,0.000000e+00
2016-06-30,3.171246e+10,1.467928e+10,1.203287e+10,1.436479e+10,2.809289e+09,1.155550e+10,2.026444e+11,2.342943e+10,1.231562e+11,5.177404e+09,...,NaN,1.141881e+10,NaN,-2.627803e+09,NaN,-1.208634e+10,NaN,2.764072e+09,NaN,1.208634e+10
2016-09-30,3.124560e+10,1.323789e+10,1.049762e+10,8.739729e+09,1.601346e+09,7.138383e+09,2.145185e+11,3.272369e+10,1.328121e+11,5.286107e+09,...,NaN,1.480282e+10,NaN,-4.744133e+09,NaN,0.000000e+00,NaN,1.784945e+09,NaN,0.000000e+00
2016-12-31,2.325585e+10,8.978965e+09,6.850105e+09,1.260923e+10,4.109281e+09,8.499948e+09,2.167187e+11,5.015198e+10,1.376864e+11,5.565005e+09,...,NaN,1.095948e+10,NaN,4.383035e+09,NaN,0.000000e+00,NaN,1.784941e+09,NaN,0.000000e+00


In [43]:
fs_df.tail(10)

,매출액,매출총이익,영업이익,법인세비용차감전순이익,법인세비용,당기순이익,자산총계,현금및현금성자산,유동자산,재고자산,...,자본총계_lag4,자산총계_lag4,자본총계_평균,자산총계_평균,GPM_ttm,OPM_ttm,NIM_ttm,ROA,ROE,payout_ratio
date,,,,,,,,,,,,,,,,,,,,,
2023-06-30,7.513055e+10,3.821114e+10,3.356226e+10,5.007187e+10,1.145140e+10,3.862047e+10,5.426777e+11,4.307478e+10,3.822631e+11,1.432822e+10,...,4.473773e+11,5.230388e+11,4.747287e+11,5.328582e+11,0.463731,0.408429,0.379902,0.189639,0.212860,NaN
2023-09-30,7.339881e+10,3.771905e+10,3.332377e+10,3.780234e+10,8.617871e+09,2.918446e+10,5.773248e+11,4.582724e+10,4.193986e+11,1.434618e+10,...,4.860192e+11,5.729324e+11,5.086785e+11,5.751286e+11,0.458181,0.400832,0.367364,0.159220,0.180020,0.0
2023-12-31,5.795259e+10,3.297029e+10,3.022980e+10,3.408138e+10,6.692618e+09,2.738876e+10,5.828695e+11,3.416588e+10,3.723729e+11,1.425245e+10,...,4.931836e+11,5.314881e+11,5.251289e+11,5.571788e+11,0.502145,0.447537,0.434017,0.199080,0.211230,0.0
2024-03-31,5.485215e+10,2.632180e+10,2.330215e+10,2.694463e+10,6.342027e+09,2.060260e+10,6.068990e+11,5.319079e+10,3.939777e+11,1.425497e+10,...,4.633885e+11,5.390183e+11,4.977892e+11,5.729587e+11,0.517431,0.460782,0.443097,0.202102,0.232621,NaN
2024-06-30,7.097924e+10,3.684863e+10,3.320962e+10,3.866305e+10,8.815810e+09,2.984724e+10,6.033247e+11,4.520417e+10,3.876482e+11,1.479359e+10,...,5.020801e+11,5.426777e+11,5.320819e+11,5.730012e+11,0.520485,0.466848,0.416136,0.186776,0.201140,NaN
2024-09-30,6.893596e+10,3.387233e+10,3.066933e+10,3.176190e+10,7.232385e+09,2.452952e+10,6.270694e+11,4.328112e+10,4.125433e+11,1.527438e+10,...,5.313378e+11,5.773248e+11,5.589996e+11,6.021971e+11,0.514455,0.464589,0.405065,0.169991,0.183127,0.0
2024-12-31,8.341884e+10,4.112332e+10,3.702024e+10,4.916950e+10,1.086981e+10,3.829968e+10,6.574438e+11,5.475925e+10,4.494981e+11,1.295931e+10,...,5.570742e+11,5.828695e+11,5.898517e+11,6.201566e+11,0.496668,0.446468,0.407206,0.182662,0.192047,0.0
2025-03-31,7.841024e+10,3.875481e+10,3.493776e+10,3.800901e+10,8.664588e+09,2.934442e+10,6.968875e+11,8.692935e+10,4.753081e+11,1.274112e+10,...,5.321898e+11,6.068990e+11,5.693267e+11,6.518933e+11,0.499095,0.450172,0.404385,0.187179,0.214325,NaN
2025-06-30,1.125221e+11,5.887732e+10,5.344436e+10,5.332249e+10,1.221226e+10,4.111023e+10,7.121667e+11,7.027449e+10,4.769625e+11,1.306151e+10,...,5.620837e+11,6.033247e+11,6.048416e+11,6.577457e+11,0.502867,0.454639,0.388258,0.202637,0.220362,NaN
